# Atelier Préparatione Données Images

Contexte

Une entreprise souhaite développer un système d’intelligence artificielle capable de reconnaître
automatiquement le type de déchet présent sur une photographie afin d'améliorer le tri des
déchets.

Le modèle devra classer chaque image dans l'une des catégories suivantes :

 cardboard : cartons ondulés, cartons plats, …
 plastic : bouteilles, emballages plastiques...
 paper : feuilles, journaux...
 glass : bouteilles et objets en verre...
 metal : canettes, boîtes métalliques...
 trash : emballages bonbons, tasses jetables, ...

Le problème est que les images collectées proviennent de plusieurs sources. Elles ne sont donc pas
homogènes : dimensions différentes ; formats différents ; images RGB et grayscale ; certaines images
sont trop petites ; certaines images sont corrompues ; quelques images sont vides ; images
dupliquées ; quelques images placées dans le mauvais dossier ; classes déséquilibrées.
L'objectif de l'atelier est donc de construire un jeu de données images propre et homogène, prêt à
être utilisé pour entraîner un modèle de Machine Learning ou de Deep Learning.

Objectifs pédagogiques

À la fin de l'atelier, l'apprenant devra être capable de :
1) explorer un dataset d'images ;
2) détecter les images problématiques ;
3) détecter les différences de résolution ;
4) détecter les différences de nombre de canaux ;
5) identifier les images trop petites ;
6) détecter les doublons ;
7) identifier les classes déséquilibrées ;
8) redimensionner les images ;
9) normaliser les valeurs des pixels ;
10) uniformiser les canaux ;
11) appliquer de la data augmentation

In [1]:
import PIL, numpy, pandas, matplotlib, imagehash, sklearn, tensorflow
print("Tout est installé correctement")

Tout est installé correctement


# Partie 1 – Exploration du dataset

##   lister les classes du dataset

In [2]:
import os  # module natif Python pour interagir avec les fichiers et dossiers du système

dossier_raw = "../data/raw"  # chemin vers le dossier qui contient les 6 sous-dossiers de classes
classes = os.listdir(dossier_raw)  # liste les noms des sous-dossiers = liste des classes

print(classes)  # affiche la liste des classes trouvées

['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


## lister le nom de chaque image 

In [7]:
import os  # module natif Python pour parcourir les fichiers et dossiers du systeme

dossier_raw = "../data/raw"  # chemin vers le dossier qui contient les 6 sous-dossiers de classes
classes = os.listdir(dossier_raw)  # liste les noms des sous-dossiers (= les classes)

noms_images = []  # liste vide qui va accueillir le nom de chaque image trouvee

for classe in classes:  # on parcourt chaque classe, une par une
    chemin_classe = os.path.join(dossier_raw, classe)  # chemin complet vers le dossier de cette classe
    for nom_fichier in os.listdir(chemin_classe):  # on parcourt chaque fichier image present dans ce dossier
        noms_images.append(nom_fichier)  # on ajoute le nom de ce fichier a la liste

print("Nombre total d'images :", len(noms_images))  # verification du nombre total de noms recuperes
print(noms_images[:5])  # affiche les 5 premiers noms, pour verifier a quoi ils ressemblent

Nombre total d'images : 1032
['cardboard1.jpg', 'cardboard10.jpg', 'cardboard100.jpg', 'cardboard101.jpg', 'cardboard102.jpg']


##  format de chaque image

In [11]:
infos_format = []  # liste qui va garder nom ET format ensemble, pour chaque image

for classe in classes:  # on parcourt chaque classe
    chemin_classe = os.path.join(dossier_raw, classe)  # chemin vers le dossier de cette classe
    for nom_fichier in os.listdir(chemin_classe):  # on parcourt chaque fichier de cette classe
        chemin_complet = os.path.join(chemin_classe, nom_fichier)  # chemin complet vers l'image
        try:  # on tente d'ouvrir l'image
            img = Image.open(chemin_complet)  # ouverture avec Pillow
            infos_format.append({"nom": nom_fichier, "format": img.format})  # on garde nom + format lies ensemble
        except Exception as e:  # si l'ouverture echoue
            pass  # on ignore ce fichier pour l'instant (deja compte comme corrompu avant)

print(infos_format[:5])  # affiche les 5 premieres paires nom/format, pour VOIR la correspondance

[{'nom': 'cardboard1.jpg', 'format': 'JPEG'}, {'nom': 'cardboard10.jpg', 'format': 'JPEG'}, {'nom': 'cardboard100.jpg', 'format': 'JPEG'}, {'nom': 'cardboard101.jpg', 'format': 'JPEG'}, {'nom': 'cardboard102.jpg', 'format': 'JPEG'}]


## recuperer le mode de chaque image

In [14]:
infos_mode = []  # liste qui va garder nom ET mode ensemble, pour chaque image

for classe in classes:  # on parcourt chaque classe
    chemin_classe = os.path.join(dossier_raw, classe)  # chemin vers le dossier de cette classe
    for nom_fichier in os.listdir(chemin_classe):  # on parcourt chaque fichier de cette classe
        chemin_complet = os.path.join(chemin_classe, nom_fichier)  # chemin complet vers l'image
        try:  # on tente d'ouvrir l'image
            img = Image.open(chemin_complet)  # ouverture avec Pillow
            infos_mode.append({"nom": nom_fichier, "mode": img.mode})  # on garde nom + mode lies ensemble
        except Exception as e:  # si l'ouverture echoue (fichier corrompu)
            pass  # on ignore ce fichier, deja compte comme corrompu a la micro-tache precedente

print("Nombre de modes recuperes :", len(infos_mode))  # verification du nombre total
print(infos_mode[:5])  # affiche les 5 premieres paires nom/mode, pour voir la correspondance

Nombre de modes recuperes : 1026
[{'nom': 'cardboard1.jpg', 'mode': 'RGB'}, {'nom': 'cardboard10.jpg', 'mode': 'RGB'}, {'nom': 'cardboard100.jpg', 'mode': 'RGB'}, {'nom': 'cardboard101.jpg', 'mode': 'RGB'}, {'nom': 'cardboard102.jpg', 'mode': 'RGB'}]


## recuperer largeur et hauteur de chaque image

In [15]:
infos_dimensions = []  # liste qui va garder nom, largeur et hauteur ensemble, pour chaque image

for classe in classes:  # on parcourt chaque classe
    chemin_classe = os.path.join(dossier_raw, classe)  # chemin vers le dossier de cette classe
    for nom_fichier in os.listdir(chemin_classe):  # on parcourt chaque fichier de cette classe
        chemin_complet = os.path.join(chemin_classe, nom_fichier)  # chemin complet vers l'image
        try:  # on tente d'ouvrir l'image
            img = Image.open(chemin_complet)  # ouverture avec Pillow
            largeur, hauteur = img.size  # .size renvoie un tuple (largeur, hauteur) en pixels
            infos_dimensions.append({"nom": nom_fichier, "largeur": largeur, "hauteur": hauteur})  # on garde les 3 infos liees ensemble
        except Exception as e:  # si l'ouverture echoue (fichier corrompu)
            pass  # on ignore ce fichier, deja compte comme corrompu

print("Nombre d'images mesurees :", len(infos_dimensions))  # verification du nombre total
print(infos_dimensions[:5])  # affiche les 5 premieres lignes, pour voir la correspondance nom/largeur/hauteur

Nombre d'images mesurees : 1026
[{'nom': 'cardboard1.jpg', 'largeur': 512, 'hauteur': 384}, {'nom': 'cardboard10.jpg', 'largeur': 512, 'hauteur': 384}, {'nom': 'cardboard100.jpg', 'largeur': 512, 'hauteur': 384}, {'nom': 'cardboard101.jpg', 'largeur': 512, 'hauteur': 384}, {'nom': 'cardboard102.jpg', 'largeur': 512, 'hauteur': 384}]
